# S003 — LSE Dividend Yield + EPS Growth
**Signal:** 40% Trailing-12M Yield Rank + 60% YoY EPS Growth Rank  
**Universe:** LSE canonical ~1,347 tickers (GBX/GBP, ≥100 GBX, ≥252d history)  
**Coverage:** ~78 tickers/day with both signals (payers with analyst EPS)  
**Portfolio:** Long-only Top-30, monthly rebalance  
**Costs:** 10 bps commission

In [ ]:
import sys, os, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

RUN_AT = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")

sys.path.insert(0, str(Path(r"c:\Personal\Business & Investments\Python codes")))
from signum import Chart
from signum.engine.dashboard import Dashboard
from signum.engine.statchart import StatChart

def _find_btest_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "AGENT_DSL_REFERENCE.md").exists():
            return p
    return Path(r"c:\Personal\Business & Investments\Python codes\btest")

BTEST_ROOT  = _find_btest_root()
os.chdir(BTEST_ROOT)

SIGNAL_ROOT = Path("research/generated/Dividend Growth/signals/003_lse_div_eps")
OUTPUTS     = SIGNAL_ROOT / "outputs"
DATA_DIR    = SIGNAL_ROOT / "data"
SHARED_DATA = Path("research/generated/Dividend Growth/shared_data")

equity  = pd.read_parquet(OUTPUTS / "equity.parquet")
returns = pd.read_parquet(OUTPUTS / "returns.parquet")
trades  = pd.read_parquet(OUTPUTS / "trades.parquet")
weights = pd.read_parquet(OUTPUTS / "weights.parquet")

raw_sum = json.loads((OUTPUTS / "summary.json").read_text())
summary = raw_sum.get("metrics", raw_sum)

for df_ in [equity, weights]:
    if hasattr(df_.index, "tz") and df_.index.tz is not None:
        df_.index = df_.index.tz_localize(None)

eq_col    = next((c for c in equity.columns  if any(k in c.lower() for k in ("nav","portfolio","equity","total"))), equity.columns[0])
strat_col = next((c for c in returns.columns if any(k in c.lower() for k in ("strategy","total","portfolio"))),     returns.columns[0])
eq        = equity[eq_col].dropna()
strat_ret = returns[strat_col].dropna()

print(f"✓  S003 outputs loaded")
print(f"   Date range : {eq.index[0].date()} → {eq.index[-1].date()}")
print(f"   Trades     : {len(trades):,}")
print(f"   Universe   : {weights.shape[1]} tickers")


In [ ]:
ann = 252
r    = strat_ret.dropna()
cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
dd   = (eq - eq.cummax()) / eq.cummax()

metrics = {
    "Total Return"    : f"{eq.iloc[-1]/eq.iloc[0]-1:.1%}",
    "CAGR"            : f"{cagr:.1%}",
    "Sharpe"          : f"{r.mean()/r.std()*np.sqrt(ann):.2f}",
    "Sortino"         : f"{r.mean()/r[r<0].std()*np.sqrt(ann):.2f}",
    "Max Drawdown"    : f"{dd.min():.1%}",
    "Calmar"          : f"{cagr/abs(dd.min()):.2f}",
    "Ann Volatility"  : f"{r.std()*np.sqrt(ann):.1%}",
    "Avg Daily Trades": f"{len(trades)/len(eq):.1f}",
}

mdf = pd.DataFrame.from_dict(metrics, orient="index", columns=["Value"])
display(mdf.style
    .set_caption("S003 — LSE Yield + EPS Growth  |  Key Metrics")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","14px"),("font-weight","bold"),("text-align","left")]},
        {"selector": "th",      "props": [("text-align","left")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace")]},
    ])
    .set_properties(**{"width": "140px"})
)


In [ ]:
nav_idx  = eq / eq.iloc[0] * 100
dd_pct   = (eq - eq.cummax()) / eq.cummax() * 100
r_sharpe = (strat_ret.rolling(63, min_periods=63).mean()
            / strat_ret.rolling(63, min_periods=63).std()
            * np.sqrt(252))

nav_df = pd.DataFrame({"time": nav_idx.index, "value": nav_idx.values})
dd_df  = pd.DataFrame({"time": dd_pct.index,  "value": dd_pct.values})
rs_df  = pd.DataFrame({"time": r_sharpe.index, "value": r_sharpe.values})

ann = 252; r = strat_ret.dropna()
cagr       = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
sharpe_val = r.mean() / r.std() * np.sqrt(ann)

Dashboard(
    panes=[
        Chart(height=280).area(nav_df, name="NAV (rebased 100)", color="#26a69a"),
        Chart(height=130).area(dd_df,  name="Drawdown %",        color="#ef5350"),
        Chart(height=130).baseline(rs_df, base_value=0, value_col="value"),
    ],
    titles=[
        f"NAV  ·  CAGR {cagr:.1%}  ·  Total Return {eq.iloc[-1]/eq.iloc[0]-1:.1%}",
        f"Drawdown  ·  Max {dd_pct.min():.1f}%",
        f"Rolling Sharpe (63d)  ·  Full-period Sharpe {sharpe_val:.2f}",
    ],
    theme="dark",
)


In [ ]:
monthly = strat_ret.resample("ME").apply(lambda x: (1+x).prod()-1)
pivot = monthly.rename_axis("date").to_frame("ret")
pivot["year"] = pivot.index.year; pivot["month"] = pivot.index.month
pivot = pivot.pivot(index="year", columns="month", values="ret") * 100
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot["Annual"] = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1) * 100

display(
    pivot.style
    .format("{:.1f}%", na_rep="")
    .background_gradient(cmap="RdYlGn", vmin=-8, vmax=8, subset=list(pivot.columns[:-1]))
    .background_gradient(cmap="RdYlGn", vmin=-20, vmax=20, subset=["Annual"])
    .set_caption("S003 — Monthly Returns (%)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center"),("min-width","48px")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace"),("min-width","48px")]},
    ])
)

ann_ret = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1)
StatChart(theme="dark", height=220, title="Annual Returns Distribution").distribution(
    ann_ret * 100, bins=14, name="Annual Return %", color="#26a69a",
    show_mean=True, show_median=True,
).show()


In [ ]:
latest_w = weights.iloc[-1].dropna()
latest_w = latest_w[latest_w > 0.001].sort_values(ascending=False) * 100

wdf = latest_w.reset_index()
wdf.columns = ["Ticker", "Weight %"]
wdf.index = range(1, len(wdf)+1)

display(
    wdf.style
    .format({"Weight %": "{:.2f}%"})
    .bar(subset=["Weight %"], color="#26a69a", vmin=0)
    .set_caption(f"S003 — Current Holdings at {weights.index[-1].date()}  ({len(wdf)} positions)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "td.col0", "props": [("font-weight","bold"),("font-family","monospace")]},
    ])
)

top5 = latest_w.head(5).index.tolist()
w_top5 = weights[top5].fillna(0) * 100
w_top5.index = pd.to_datetime(w_top5.index).normalize()
w_top5_df = w_top5.reset_index()
w_top5_df.columns = ["time"] + top5

chart = Chart(height=220, theme="dark", watermark="Top-5 Weights over Time")
for tk in top5:
    chart.line(w_top5_df[["time", tk]].rename(columns={tk: "value"}), name=tk)
chart


In [ ]:
composite = pd.read_parquet(DATA_DIR / "composite.parquet")
if hasattr(composite.index, "tz") and composite.index.tz is not None:
    composite.index = composite.index.tz_localize(None)

n_scored = composite.notna().sum(axis=1).resample("ME").mean()
n_held   = (weights > 0.001).sum(axis=1).resample("ME").mean()

Dashboard(
    panes=[
        Chart(height=180).area(
            pd.DataFrame({"time": n_scored.index, "value": n_scored.values}),
            name="Tickers scored", color="#1976d2"),
        Chart(height=130).area(
            pd.DataFrame({"time": n_held.index, "value": n_held.values}),
            name="Positions held", color="#ff9800"),
    ],
    titles=[
        f"Signal Coverage (payers w/ analyst EPS)  ·  avg {n_scored.mean():.0f} tickers/month",
        f"Avg Positions Held  ·  avg {n_held.mean():.0f}",
    ],
    theme="dark",
)


---

## Per-Ticker Attribution

Decompose portfolio returns by individual ticker using daily `weight × price return`. Shows which names drove performance, how long each was held, and trade-level realized P&L.

In [ ]:
prices_long = pd.read_parquet(SHARED_DATA / "lse_prices.parquet")
prices_wide = prices_long.pivot_table(index="date", columns="ticker", values="close")
price_ret   = prices_wide.sort_index().pct_change(fill_method=None).clip(-0.5, 0.5)

w = weights.fillna(0.0)
w.index = pd.to_datetime(w.index).normalize()
common_tickers = w.columns.intersection(price_ret.columns)
common_dates   = w.index.intersection(price_ret.index)

w_sub = w.loc[common_dates, common_tickers]
r_sub = price_ret.loc[common_dates, common_tickers].fillna(0.0)

daily_contrib = w_sub.shift(1).fillna(0.0) * r_sub
ever_held     = (w_sub > 0.001).any()
contrib       = daily_contrib.loc[:, ever_held].sum()
n_days_held   = (w_sub.loc[:, ever_held] > 0.001).sum()
avg_wt        = w_sub.loc[:, ever_held].where(w_sub.loc[:, ever_held] > 0.001).mean()

attr = pd.DataFrame({
    "contrib_bps"    : (contrib * 10000).round(1),
    "contrib_pct"    : (contrib * 100).round(2),
    "days_held"      : n_days_held,
    "pct_time_held"  : (n_days_held / len(w_sub) * 100).round(1),
    "avg_weight_pct" : (avg_wt * 100).round(2),
}).sort_values("contrib_bps", ascending=False)

print(f"Tickers held: {ever_held.sum()} | Attribution sum: {contrib.sum()*100:.2f}% | Portfolio total: {(eq.iloc[-1]/eq.iloc[0]-1)*100:.2f}%")

display(attr.head(15).style
    .background_gradient(subset=["contrib_bps"], cmap="Greens")
    .format({"contrib_pct":"{:.2f}%","pct_time_held":"{:.1f}%","avg_weight_pct":"{:.2f}%"})
    .set_caption("Top 15 Contributors"))
display(attr.tail(10).style
    .background_gradient(subset=["contrib_bps"], cmap="Reds_r")
    .format({"contrib_pct":"{:.2f}%","pct_time_held":"{:.1f}%","avg_weight_pct":"{:.2f}%"})
    .set_caption("Bottom 10 Detractors"))


In [ ]:
top10_tickers = attr.head(10).index.tolist()
bot5_tickers  = attr.tail(5).index.tolist()

cum_contrib_top = daily_contrib[top10_tickers].cumsum() * 10000
cum_contrib_bot = daily_contrib[bot5_tickers].cumsum() * 10000
cum_contrib_top.index = pd.to_datetime(cum_contrib_top.index).normalize()
cum_contrib_bot.index = pd.to_datetime(cum_contrib_bot.index).normalize()

chart_top = Chart(height=240, theme="dark", watermark="Top-10 Cumulative Contribution (bps)")
for tk in top10_tickers:
    chart_top.line(cum_contrib_top[[tk]].reset_index().rename(columns={"index":"time", tk:"value"}), name=tk)
chart_top.show()

chart_bot = Chart(height=200, theme="dark", watermark="Bottom-5 Cumulative Contribution (bps)")
for tk in bot5_tickers:
    chart_bot.line(cum_contrib_bot[[tk]].reset_index().rename(columns={"index":"time", tk:"value"}), name=tk)
chart_bot.show()


In [ ]:
scatter_df = attr[attr["days_held"] > 0].copy()
StatChart(theme="dark", height=380, title="Days Held vs Cumulative Contribution (bps)").scatter(
    scatter_df["days_held"].values,
    scatter_df["contrib_bps"].values,
    labels=scatter_df.index.tolist(),
    name="Ticker",
    x_label="Days Held",
    y_label="Contribution (bps)",
).show()


In [ ]:
ts = (
    trades.groupby("instrument")
    .agg(
        n_buys        =("side", lambda x: (x=="BUY").sum()),
        n_sells       =("side", lambda x: (x=="SELL").sum()),
        total_notional=("notional", "sum"),
        realized_pnl  =("realized_pnl", "sum"),
        commission    =("commission", "sum"),
    )
)
ts["net_pnl"] = ts["realized_pnl"] - ts["commission"]
ts = ts.sort_values("net_pnl", ascending=False)

print(f"Unique tickers: {len(ts)} | Total commission: {trades['commission'].sum():,.0f} | Realized P&L: {trades['realized_pnl'].sum():,.0f}")

display(ts.head(15).round(1).style
    .background_gradient(subset=["net_pnl"], cmap="RdYlGn")
    .format({"total_notional":"{:,.0f}","realized_pnl":"{:,.1f}","commission":"{:,.1f}","net_pnl":"{:,.1f}"})
    .set_caption("Top 15 by Net Realized P&L"))
display(ts.tail(10).round(1).style
    .background_gradient(subset=["net_pnl"], cmap="RdYlGn")
    .format({"total_notional":"{:,.0f}","realized_pnl":"{:,.1f}","commission":"{:,.1f}","net_pnl":"{:,.1f}"})
    .set_caption("Bottom 10 by Net Realized P&L"))
